# From replay records to time observable features

Md Forhadul Islam, London

This is my independent Dota 2 adaptation of the approach of Fazilat Hojaji,
Robert E. McIlroy, Antoine Dupuy, Gianpaolo Pedroni, Adam J. Toth and Mark J. Campbell.
Their original League of Legends paper is https://doi.org/10.1016/j.chbr.2025.100718.
It is not an exact reproduction of their numerical results. The Betty dataset is
published by wolframko using Valve replays and STRATZ data. Sources, licences and
the literature tracker are recorded in the project documentation.

I explain each step before the code and interpret the output afterwards. All
displayed experimental results come from saved calculations. Reading them does
not train a new model or choose an approach using the final test outcomes.


## Find the saved study

A path tells Python where a file lives. JSON stores labelled evidence. A DataFrame is a table with named columns. This setup reads existing results; it cannot change trained weights or the test partition.

In [1]:
from pathlib import Path  # Represent folders consistently on Windows and other systems.
import sys  # Make the shared project modules available to this notebook.
import json  # Read saved evidence with its original field names.
import pandas as pd  # Display numerical records as labelled tables.
from IPython.display import display, Image, Markdown  # Show readable tables, figures and notes.
ROOT = Path.cwd().resolve()  # Begin from the notebook execution folder.
if ROOT.name == "notebooks":  # Jupyter normally starts within the notebook folder.
    ROOT = ROOT.parent  # Move up once to reach the shared project directory.
assert (ROOT / "src").is_dir(), "Open this notebook from the project or notebooks folder."
sys.path.insert(0, str(ROOT / "src"))  # Import the same scientific code used by terminal runs.
def evidence(filename):  # Keep evidence loading consistent across the notebook.
    return json.loads((ROOT / "outputs/results" / filename).read_text(encoding="utf-8"))  # Decode one saved result.
summary = evidence("D_summary.json")  # Read the completed study without recomputing final predictions.


## Why are the clocks and teams important?

A replay clock can keep advancing during a pause. A match clock describes playing time. Confusing them would select the wrong state at a nominal prediction time. A player slot is also not a reliable team label in these files. The pipeline aligns combat event timestamps to preceding states and maps hero identities to the metadata draft.

In [2]:
clock = evidence("D_clock_alignment.json")  # Read the audit that motivated event anchored timing.
identity = evidence("D_identity_audit.json")  # Read the evidence behind hero based team assignment.
print("Clock audit fields:", list(clock))  # Expose the audit structure for further inspection.
print("Identity audit fields:", list(identity))  # Keep the team mapping audit discoverable.
display(pd.Series(summary["integrity"], name="Integrity checks"))  # Show checked counts and acknowledged limits.

Clock audit fields: ['version', 'anchors', 'matches', 'event_shards', 'sampled_replay_minus_match_seconds_quantiles', 'anchor_max_age_match_seconds', 'state_rule', 'native_tick_rate', 'limitation']
Identity audit fields: ['version', 'snapshot_groups_before', 'snapshot_groups_valid', 'unknown_name_counts', 'reason', 'excluded']


metadata_draft_complete_checks                                                                     9071
metadata_draft_mismatches                                                                             0
metadata_draft_checks_unavailable                                                                   314
ancient_event_outcome_checks                                                                       9385
ancient_event_outcome_mismatches                                                                      0
ancient_timestamp_minus_duration_quantiles_seconds            [-0.0334, 0.2001, 0.4667, 0.7002, 0.9668]
snapshots_checked                                                                                 21040
no_completed_match_snapshots                                                                       True
all_match_observations_share_partition                                                             True
partition_fingerprint_unchanged                                 

The checks establish internal consistency, not independent ground truth. States strictly precede the selected event anchor. The anchor and state tolerances together give a conservative 2.4 second timing bound. The older max_snapshot_age_seconds field describes only the 1.2 second state matching component; it must not be read as the total bound.

## What is a feature and what is the target?

A feature is information available at prediction time. The target is the later outcome we try to predict. Each feature is a currently observed Radiant team total minus the corresponding Dire total. Positive values mean Radiant has more of that statistic. Net worth is economic resources; experience measures progression; a deny prevents an opponent collecting a friendly unit reward.

In [3]:
preparation = summary["preparation"]  # Read the exact approved feature list.
display(pd.DataFrame({"feature": preparation["features"]}))  # Show every predictor actually used.
display(pd.DataFrame(preparation["rows_by_horizon_partition"]).pivot(index="horizon_seconds", columns="partition", values="matches"))  # Show how eligibility varies over time.
print("Target:", preparation["target"])  # Name the later result rather than leaving the label implicit.

,feature
0,net_worth_diff
1,xp_diff
2,kills_diff
3,assists_diff
4,last_hits_diff
5,denies_diff
6,level_diff
7,hp_diff
8,is_alive_diff
9,towers_alive_diff


partition,calibration,test,train,validation
horizon_seconds,,,,
600,443,1745,5243,1281
1200,442,1729,5095,1290
1800,177,758,2251,586


Target: radiant_win


Final duration, final statistics, identifiers and retrospective roles are excluded as predictors. Duration is used only to exclude already completed matches from a live prediction cohort. Towers are available; barracks and trustworthy ward locations are not used. Health and alive counts describe current state, not future survival.

## Why split by chronology and whole match?

Training fits parameters, validation selects checkpoints, calibration adjusts confidence, and the final test estimates later performance after decisions are fixed. Every observation from one match remains in one partition. Whole UTC dates are grouped because exact series identifiers are absent.

In [4]:
protocol = summary["protocol"]  # Read the membership fingerprint and frozen settings.
display(pd.Series(protocol["partition_counts"], name="All metadata matches"))  # These counts precede horizon eligibility filtering.
print("UTC boundaries:", protocol["date_boundaries"])  # Show the chronological cut points.
print("Frozen partition SHA256:", protocol["partition_sha256"])  # Identify the exact saved split.
print("Final status:", protocol["status"])  # Make the evaluation boundary explicit.

train          5620
test           1890
validation     1390
calibration     485
Name: All metadata matches, dtype: int64

UTC boundaries: ['2025-08-20 00:00:00+00:00', '2025-09-29 00:00:00+00:00', '2025-10-14 00:00:00+00:00']
Frozen partition SHA256: e8289bea726c74ec18e6517f6398cc37482182f833f5e6b1777aff1c7ccbe6c4
Final status: final evaluation complete; do not tune on these results


The tables differ because some games are finished or cannot be safely observed at a later horizon. Teams may recur across dates. UTC day grouping is a partial dependence control, not an unseen team or unseen tournament evaluation.

## Inspect the prepared table without rebuilding it

The expensive extraction is implemented in src/dota_data.py. This cell reads its saved table if raw data have been prepared locally. Published notebooks retain their outputs even when a reader has not downloaded the dataset.

In [5]:
snapshot_path = ROOT / "data/processed/dota_snapshots.parquet"  # Locate the cached feature table.
if snapshot_path.exists():  # Permit a lightweight read on the original study computer.
    snapshots = pd.read_parquet(snapshot_path)  # Load already extracted observations.
    assert snapshots.groupby("match_id").partition.nunique().max() == 1  # Check that repeated times never cross partitions.
    display(snapshots.drop(columns=["radiant_win"], errors="ignore").head())  # Inspect inputs without using outcomes to choose a method.
else:  # A clean clone intentionally excludes raw and processed data.
    print("Run the documented download and preparation commands to inspect local rows.")  # Explain the missing cache honestly.

,match_id,horizon_seconds,net_worth_diff,xp_diff,kills_diff,assists_diff,last_hits_diff,denies_diff,level_diff,hp_diff,is_alive_diff,towers_alive_diff,start_time,partition
0,7581685965,600,-3528,-2147,-5,-7,-31,2,-3,-99,0,0,1707667950,train
1,7581685965,1200,-6135,-8465,-5,-2,-181,1,-8,-738,0,0,1707667950,train
2,7581685965,1800,-14822,-20399,-4,-4,-407,2,-11,-1822,-1,-2,1707667950,train
3,7581799243,600,3666,2411,5,8,48,30,3,-583,-1,0,1707671730,train
4,7581799243,1200,2781,1952,4,1,82,45,2,1299,3,1,1707671730,train


Preparation can be reproduced using the commands in the README. I keep the original files separate from derived tables so that a transformation never silently changes source data.

## Read the shared implementation line by line

The source below is the actual function used by the pipeline, including comments explaining what each operation does and why. `inspect` reads Python source; it does not call the function or refit a model. A function is a named sequence of reusable steps; a class groups a model's state and prediction methods. Read the comments alongside the preceding explanation, then open the source file to explore further. The frozen result record retains the original implementation fingerprints; later documentation changes do not replace that historical record.

In [6]:
import inspect  # Read implementation text without running a scientific experiment.
from IPython.display import Code  # Render Python source with readable syntax highlighting.
import dota_data  # Access the same shared module used by the terminal pipeline.
display(Code(inspect.getsource(dota_data.latest_rows), language='python'))  # Show the commented implementation without calling it.

def latest_rows(path, columns, keys, anchors=None):
    """Read only rows immediately preceding each fixed observation time.

    Missing observations are excluded, never borrowed from the future.
    Parquet filtering reduces memory use, although file scanning still costs time.
    """
    if anchors is None:  # Tests may supply a small independent clock table.
        anchors = pd.read_parquet(PROCESSED / "dota_clock_cutoffs.parquet")  # Use combat log match time rather than the replay elapsed field.
    identifiers = pq.read_table(path, columns=["match_id"]).column("match_id").unique().to_pylist()  # Identify the matches in this source shard.
    anchors = anchors.loc[anchors.match_id.isin(identifiers)]  # Keep only cutoffs relevant to this shard.
    columns = list(dict.fromkeys(columns + ["tick"]))  # Include raw ticks so the temporal rule can be checked.
    filters = [[("match_id", "==", int(row.match_id)), ("tick", ">=", int(row.cutoff_tick) - 36), ("tick", "<", int(row.cutoff_tick))] for row in anchors.itertuples()]  # Strictly precede an event tick known to be no later than the requested match time.
    if not filters:  # Some shards may have no sufficiently recent event anchors at these horizons.
        return pd.DataFrame(columns=columns + ["horizon_seconds"])  # Return an empty compatible table rather than future observations.
    table = pq.read_table(path, columns=columns, filters=filters)  # Read the requested columns and eligible times.
    frame = table.to_pandas()  # Work on the small filtered table, not the entire source.
    output = []  # Collect one table for each time horizon.
    for horizon in HORIZONS:  # Keep time horizons separate so they cannot leak into each other.
        cutoff = anchors.loc[anchors.horizon_seconds == horizon, ["match_id", "cutoff_tick"]]  # Select this match time's safely preceding event tick.
        eligible = frame.merge(cutoff, on="match_id", validate="many_to_one")  # Attach the cutoff without using outcomes.
        eligible = eligible.loc[(eligible.tick < eligible.cutoff_tick) & (eligible.tick >= eligible.cutoff_tick - 36)].copy()  # Keep preceding states at most one native interval behind the safe anchor.
        eligible = eligible.sort_values("tick").drop_duplicates(keys, keep="last")  # Choose the last eligible raw state per player or building.
        eligible["horizon_seconds"] = horizon  # Preserve which observation time this row represents.
        output.append(eligible)  # Retain the rows for later team aggregation.
    return pd.concat(output, ignore_index=True)  # Combine horizons while retaining their labels.